# 1. Preparing a modelling table

## Goal

Parse LiqCryst-style phase sequences, inspect quality flags, calculate
Mordred descriptors, separate chiral phases.


### 1. Locate the example and output files

**Extension.** A laboratory data folder may be represented by another relative path or supplied through a configuration file.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
EXAMPLE_DATA = PROJECT_ROOT / "examples" / "data"
PAPER_DATA = PROJECT_ROOT / "paper" / "data" / "processed"
MODEL_DIRECTORY = PROJECT_ROOT / "models"
RESULT_DIRECTORY = PROJECT_ROOT / "results"
RESULT_DIRECTORY.mkdir(parents=True, exist_ok=True)

RAW_FILE = EXAMPLE_DATA / "sample.csv"
PREPARED_FILE = RESULT_DIRECTORY / "prepared_sample.parquet"
print("Input:", RAW_FILE)
print("Output:", PREPARED_FILE)

Input: ../examples/data/sample.csv
Output: ../results/prepared_sample.parquet


### 2. Read and inspect the raw records

**Caution.** The parser requires `SMILES` and `Phases`; an `ID` column is strongly recommended for provenance.

**Extension.** Additional experimental metadata may remain in the table and will pass through unchanged.

In [2]:
from IPython.display import display

from lc_qspr.preprocessing import read_table

raw_table = read_table(RAW_FILE)
display(raw_table)

,SMILES,Phases
ID,,
0,C1C=CC=CC=1,Cr 6 is
3,CCCCCCCCCC1C=C3C(=CC=1)C2C(=CC(=CC=2)C(=O)C)C3,Cr 64 A 81 is
4,CCCCCCCCCC1C=C3C(=CC=1)C2C(=CC(=CC=2)C(=O)CCCC...,Cr 37 A 38 N 40 is
5,CCCCCC1C=CC(=CC=1)C2C=CC(OCCCC)=CC=2,Cr 63 B 81 A 89 is
6,O=C(OCCCCO)C1C=CC(=CC=1)C(OCCCCO)=O,Cr 65 N 72 is


### 3. Parse phase classes and transition temperatures

**Caution.** Rows containing unsupported phase labels, missing neighbouring temperatures or re-entrant phases are flagged rather than silently discarded.

**Extension.** The returned report can be archived with an experimental data release as a compact audit record.

In [3]:
from lc_qspr.preprocessing import parse_phase_table

parsed_table, parse_report = parse_phase_table(raw_table)
print(parse_report)
display(parsed_table[[
    "SMILES", "Phases", "exclude", "reentrant",
    "Melting", "Clearing", "Ntype", "Atype", "Btype",
]])

PhaseParseReport(rows=5, excluded=0, reentrant=0, phase_labels={'Cr': 5, 'is': 5, 'A': 3, 'N': 2, 'B': 1})


,SMILES,Phases,exclude,reentrant,Melting,Clearing,Ntype,Atype,Btype
ID,,,,,,,,,
0,C1C=CC=CC=1,Cr 6 is,False,False,6.0,6.0,0,0,0
3,CCCCCCCCCC1C=C3C(=CC=1)C2C(=CC(=CC=2)C(=O)C)C3,Cr 64 A 81 is,False,False,64.0,81.0,0,1,0
4,CCCCCCCCCC1C=C3C(=CC=1)C2C(=CC(=CC=2)C(=O)CCCC...,Cr 37 A 38 N 40 is,False,False,37.0,40.0,1,1,0
5,CCCCCC1C=CC(=CC=1)C2C=CC(OCCCC)=CC=2,Cr 63 B 81 A 89 is,False,False,63.0,89.0,0,1,1
6,O=C(OCCCCO)C1C=CC(=CC=1)C(OCCCCO)=O,Cr 65 N 72 is,False,False,65.0,72.0,1,0,0


### 4. Calculate Mordred descriptors



In [4]:
from lc_qspr.preprocessing import add_mordred_descriptors

descriptor_table = add_mordred_descriptors(parsed_table)
print(f"Prepared {descriptor_table.shape[0]} rows and {descriptor_table.shape[1]} columns.")
display(descriptor_table.iloc[:, :12])

Prepared 5 rows and 1657 columns.


,SMILES,Phases,reentrant,exclude,Ntype,Nm,Np,Atype,Am,Ap,Btype,Bm
ID,,,,,,,,,,,,
0,C1C=CC=CC=1,Cr 6 is,False,False,0,NaN,NaN,0,NaN,NaN,0,NaN
3,CCCCCCCCCC1C=C3C(=CC=1)C2C(=CC(=CC=2)C(=O)C)C3,Cr 64 A 81 is,False,False,0,NaN,NaN,1,64.0,81.0,0,NaN
4,CCCCCCCCCC1C=C3C(=CC=1)C2C(=CC(=CC=2)C(=O)CCCC...,Cr 37 A 38 N 40 is,False,False,1,38.0,40.0,1,37.0,38.0,0,NaN
5,CCCCCC1C=CC(=CC=1)C2C=CC(OCCCC)=CC=2,Cr 63 B 81 A 89 is,False,False,0,NaN,NaN,1,81.0,89.0,1,63.0
6,O=C(OCCCCO)C1C=CC(=CC=1)C(OCCCCO)=O,Cr 65 N 72 is,False,False,1,65.0,72.0,0,NaN,NaN,0,NaN


### 5. Separate chiral phases and apply quality filters


In [5]:
from lc_qspr.preprocessing import filter_training_table, separate_chiral_phases

separated_table, chiral_phases = separate_chiral_phases(descriptor_table)
training_table = filter_training_table(
    separated_table,
    minimum_nitrogen_and_carbon=0,
    suspicious_tokens=(),
)
print("Chiral phases found:", chiral_phases or "none in this example")
print("Rows after filtering:", len(training_table))

Chiral phases found: none in this example
Rows after filtering: 5


### 6. Save and read back the prepared table

**Caution.** Write new artefacts to `results/`.

CSV is more portable, whilst Parquet is faster and preserves numeric types more reliably for the full descriptor database.

In [6]:
from lc_qspr.preprocessing import write_table

write_table(training_table, PREPARED_FILE)
reloaded_table = read_table(PREPARED_FILE)
assert reloaded_table.shape == training_table.shape
assert reloaded_table.index.tolist() == training_table.index.tolist()
print("Round-trip check passed:", reloaded_table.shape)

Round-trip check passed: (5, 1657)
